# Valores faltantes {#sec-missing-values}

## Introducción

En este capítulo veremos las herramientas y trucos para lidiar con valores faltantes. Empezaremos hablando de algunas herramientas generales para trabajar con valores faltantes registrados como `NA`. Luego exploraremos la idea de los valores faltantes implícitos, valores que simplemente están ausentes de tus datos, y mostraremos algunas herramientas que puedes usar para hacerlos explícitos.
Terminaremos con una discusión relacionada sobre los grupos vacíos, causados por categorías que no aparecen en los datos.

In [ ]:
# remove cell
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline

# Plot settings
plt.style.use("https://github.com/aeturrell/python4DS/raw/main/plot_style.txt")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

### Requisitos previos

Este capítulo usará el paquete de análisis de datos **pandas**.

## Valores faltantes explícitos

Para empezar, exploremos algunas herramientas útiles para crear o eliminar valores faltantes explícitos, es decir, celdas donde ves un `NA` o `nan`.

### Tipos de valores faltantes

Algo a tener en cuenta sobre los valores faltantes en **pandas** es que ¡no todos son iguales!

Por ejemplo, los números reales en **pandas** (como el dtype `float64`) usan un 'nan' (es decir, Not a Number, «no es un número»):

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame([5, 27.3, np.nan, -16], columns=["numbers"])
df

¡Pero esta no es la única forma! También podemos usar el valor `None` integrado en Python (que, aquí, se convierte en NaN porque los valores válidos son todos números de punto flotante) y el `pd.NA` de **pandas**:

In [ ]:
numbers = pd.DataFrame([pd.NA, 27.3, np.nan, -16, None], columns=["numbers"])
numbers

Sin embargo, con el tipo de dato object (el predeterminado para strings), los tipos pueden coexistir:

In [ ]:
fruits = pd.DataFrame(
    ["orange", np.nan, "apple", None, "banana", pd.NA], columns=["fruit"]
)
fruits

Ambos tipos de valor faltante se pueden encontrar con la función `.isna()` de **pandas**. Esta devuelve una nueva columna de valores boolean que son `True` si el valor es cualquier tipo de valor faltante.

In [ ]:
fruits.isna()

Por comodidad, también existe una función `notna()`.

### Cómo tratar los valores faltantes explícitos

Hay varias opciones para tratar los valores faltantes. La función `fillna()` permite hacerlo. Veámosla con algunos datos de prueba:

In [ ]:
nan_df = pd.DataFrame(
    [
        [np.nan, 2, None, 0],
        [3, 4, np.nan, 1],
        [5, np.nan, np.nan, pd.NA],
        [np.nan, 3, np.nan, 4],
    ],
    columns=list("ABCD"),
)

nan_df

Primero, podemos simplemente rellenar cualquier valor faltante con un único valor fijo:

In [ ]:
nan_df.fillna(0)

Esto se puede hacer columna por columna; aquí reemplazamos todos los elementos NaN de las columnas ‘A’, ‘B’, ‘C’ y ‘D’ por 0, 1, 2 y 3 respectivamente.

In [ ]:
nan_df.fillna(value={"A": 0, "B": 1, "C": 2, "D": 3})

También podemos propagar los valores no nulos hacia adelante o hacia atrás (en relación con el índice)

In [ ]:
nan_df.fillna(method="ffill")

In [ ]:
nan_df.fillna(method="bfill")

Las opciones de relleno hacia adelante y hacia atrás son especialmente útiles para series temporales, ¡pero ten cuidado al usarlas si estás haciendo un ejercicio de pronóstico!

Otra característica de todas estas funciones es que puedes limitar el número de NaN que se reemplazan con el argumento de palabra clave `limit=`.

In [ ]:
nan_df.fillna(value={"A": 0, "B": 1, "C": 2, "D": 3}, limit=1)

Por supuesto, otra opción podría ser simplemente filtrar por completo los valores faltantes. Hay un par de formas de hacerlo según si quieres eliminar filas enteras (`axis=0`) o columnas (`axis=1`; aunque en este caso, como hay al menos un NaN en cada columna, ¡no quedará ningún dato!)

In [ ]:
nan_df["A"].dropna(axis=0)  # on a single column

In [ ]:
nan_df.dropna(axis=1)

`dropna()` también admite algunos argumentos de palabra clave; por ejemplo, `how="all"` solo elimina una columna o fila si *todos* sus valores son NA.

In [ ]:
nan_df.dropna(how="all")

Existe una palabra clave `thresh` (de threshold, «umbral»), que te permite conservar solo las filas o columnas que contienen como máximo cierto número de observaciones faltantes.

Otra forma de filtrar los nan es usar los mismos métodos de filtrado que usarías normalmente, mediante columnas boolean, en combinación con la función `.notna()`. En el ejemplo siguiente, vemos todas las columnas de las filas en las que A no es NA.

In [ ]:
nan_df[nan_df["A"].notna()]

### Añadir valores NA

A veces te encontrarás con el problema opuesto, en el que algún valor concreto en realidad representa un valor faltante. Esto suele ocurrir en datos generados por software antiguo que no tiene una forma adecuada de representar valores faltantes, por lo que debe usar algún valor especial como 99 o -999.

Si es posible, resuélvelo al leer los datos, por ejemplo, usando el argumento de palabra clave `na_values=` al llamar a `pd.read_csv()`. Si descubres el problema más tarde, o tu fuente de datos no ofrece una forma de manejarlo al leer el archivo, puedes usar diversas opciones para reemplazar los datos dados:

In [ ]:
stata_df = pd.DataFrame([[3, 4, 5], [-7, 4, -99], [-99, 6, 5]], columns=list("ABC"))

stata_df

La opción más sencilla probablemente sea `.replace()`:

In [ ]:
stata_df.replace({-99: pd.NA})

Como `.replace()` acepta un diccionario, es posible reemplazar varios valores a la vez:

In [ ]:
stata_df.replace({-99: pd.NA, -7: pd.NA})

Ten en cuenta que esto se aplica a *todas* las columnas del dataframe. Para aplicarlo a una sola, simplemente selecciona esa columna específica.

## Valores faltantes implícitos

Hasta ahora hemos hablado de valores que faltan **explícitamente**, es decir, puedes ver un `NA` o algo similar en tus datos.
Pero los valores también pueden faltar **implícitamente**, si una fila entera de datos simplemente está ausente.
Ilustremos la diferencia con un dataset sencillo que registra el precio de una acción cada trimestre:

In [ ]:
stocks = pd.DataFrame(
    {
        "year": [2020, 2020, 2020, 2020, 2021, 2021, 2021],
        "qtr": [1, 2, 3, 4, 2, 3, 4],
        "price": [1.88, 0.59, 0.35, np.nan, 0.92, 0.17, 2.66],
    }
)
stocks

Este dataset tiene dos observaciones faltantes:

-   El `price` del cuarto trimestre de 2020 falta explícitamente, porque su valor es `NA`.

-   El `price` del primer trimestre de 2021 falta implícitamente, porque simplemente no aparece en el dataset.

Una forma de pensar en la diferencia es con este koan de estilo zen:

> Un valor faltante explícito es la presencia de una ausencia.
>
> Un valor faltante implícito es la ausencia de una presencia.

A veces quieres hacer explícitos los faltantes implícitos para tener algo físico con lo que trabajar.
En otros casos, la estructura de los datos te impone faltantes explícitos y quieres deshacerte de ellos.
Las siguientes secciones tratan algunas herramientas para pasar de faltantes implícitos a explícitos y viceversa.

### Pivotar

Ya has visto una herramienta que puede hacer explícitos los faltantes implícitos y viceversa: el pivot. Hacer los datos más anchos puede volver explícitos los valores faltantes implícitos, porque cada combinación de filas y nuevas columnas debe tener algún valor. Por ejemplo, si pivotamos `stocks` para poner `quarter` en las columnas (y hacemos de `year` el índice), ambos valores faltantes se vuelven explícitos:

In [ ]:
stocks.pivot(columns="qtr", values="price", index="year")

Por defecto, hacer los datos más largos conserva los valores faltantes explícitos.

### Valores faltantes en variables categóricas

Un último tipo de ausencia es el grupo vacío, un grupo que no contiene ninguna observación, lo cual puede surgir al trabajar con datos categóricos. 

Por ejemplo, imagina que tenemos un dataset que contiene información de salud sobre algunas personas:

In [ ]:
health = pd.DataFrame(
    {
        "name": ["Ikaia", "Oletta", "Leriah", "Dashay", "Tresaun"],
        "smoker": ["no", "no", "previously", "no", "yes"],
        "age": [34, 88, 75, 47, 56],
    }
)
health["smoker"] = health["smoker"].astype("category")

Ahora eliminamos la última fila de datos:

In [ ]:
health_cut = health.iloc[:-1, :]
health_cut

El valor 'yes' de smoker ahora no aparece (aparentemente) en ningún lugar de nuestro dataframe. Pero si ejecutas `value_counts()` para contar cuántos hay de cada categoría, verás que el dataframe 'recuerda' que existe una categoría 'yes' que actualmente no está presente:

In [ ]:
health_cut["smoker"].value_counts()

Verás que ocurre lo mismo con una operación `groupby()`:

In [ ]:
health_cut.groupby("smoker")["age"].mean()

Aquí puedes ver que, como calculamos la media de un número que no existe, obtuvimos un NaN en lugar de un valor real para la fila yes (pero sí hay una fila 'yes').